## Data Preperation

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier


In [2]:
PROJECT_ROOT = Path.cwd()

for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv")

df = pd.read_csv(DATA_PATH)

print(DATA_PATH)
print(df.shape)
df.head()

/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv
(69987, 56)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,diabetesMed,readmitted,readmitted_30,hba1c_group,primary_diagnosis,age_group,discharge_group,race_group,admission_source_group,medical_specialty_group
0,24437208,135,Caucasian,Female,[50-60),2,1,1,8,Cardiology,...,Yes,<30,1,No test was performed,Circulatory,30-60,Home,Caucasian,Physician/clinic referral,Cardiology
1,29758806,378,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,...,No,NO,0,No test was performed,Musculoskeletal,30-60,Home,Caucasian,Physician/clinic referral,Surgery
2,189899286,729,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,...,Yes,NO,0,Normal result of the test,Injury,>60,Other,Caucasian,Emergency room,Internal Medicine
3,64331490,774,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,...,Yes,NO,0,"High, medication changed",Other,>60,Home,Caucasian,Emergency room,Internal Medicine
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,...,Yes,NO,0,No test was performed,Genitourinary,30-60,Home,AfricanAmerican,Emergency room,Internal Medicine


In [3]:
target_col = "readmitted_30"

drop_cols = [
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
]

# Drop columns only if they actually exist
drop_cols_existing = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols_existing)
y = df[target_col]

print(X.shape)
print(y.value_counts())
print(y.value_counts(normalize=True))

(69987, 52)
readmitted_30
0    63702
1     6285
Name: count, dtype: int64
readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, y_train.mean())
print("Test:", X_test.shape, y_test.mean())

Train: (55989, 52) 0.08980335423029524
Test: (13998, 52) 0.08979854264894985


In [5]:
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()

print("Categorical features:", len(categorical_features))
print(categorical_features)

print("Numeric features:", len(numeric_features))
print(numeric_features)

Categorical features: 41
['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'hba1c_group', 'primary_diagnosis', 'age_group', 'discharge_group', 'race_group', 'admission_source_group', 'medical_specialty_group']
Numeric features: 11
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']


In [6]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features)
    ]
)

In [ ]:
def save_confusion_matrix(model, X_test, y_test, model_name, output_dir):
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual not readmitted", "Actual readmitted"],
        columns=["Predicted not readmitted", "Predicted readmitted"]
    )

    file_name = (
        model_name
        .lower()
        .replace(":", "")
        .replace(" ", "_")
        .replace("/", "_")
    )

    cm_df.to_csv(output_dir / f"confusion_matrix_{file_name}.csv")

    tn, fp, fn, tp = cm.ravel()

    total = tn + fp + fn + tp

    cm_long = pd.DataFrame([{
        "model": model_name,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "total": total,
        "true_negative_rate": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "false_positive_rate": fp / (tn + fp) if (tn + fp) > 0 else 0,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0,
        "true_positive_rate_recall": tp / (fn + tp) if (fn + tp) > 0 else 0
    }])

    return cm_df, cm_long